# AIN58XS — Artificial Neural Network Analysis
**Cape Peninsula University of Technology**  
Examiner: A. Wyngaard | Moderator: V. Moyo | External Moderator: A. Yusuf  
Due: May 2026

---
This notebook explores and analyses an ANN for identifying handwritten digits (MNIST dataset).  
Architecture: **784 input → hidden layers → 10 output (softmax)**

## 0. Install & Import Dependencies

In [ ]:
# Install required packages (run once)
# !pip install tensorflow scikit-learn seaborn matplotlib numpy pillow scipy

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import confusion_matrix, classification_report
from scipy import ndimage                          # for centre-of-mass centring

print(f"TensorFlow version : {tf.__version__}")
print(f"NumPy version      : {np.__version__}")


---
## 1a. Load MNIST & Examine Dimensions
> **Question a:** What are the dimensions of the MNIST dataset?

In [ ]:
# Load the MNIST dataset
(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = mnist.load_data()

print("═" * 50)
print("  MNIST Dataset Dimensions (Question a)")
print("═" * 50)
print(f"  Training images shape : {x_train_raw.shape}")
print(f"  Training labels shape : {y_train_raw.shape}")
print(f"  Test images shape     : {x_test_raw.shape}")
print(f"  Test labels shape     : {y_test_raw.shape}")
print(f"  Pixel value range     : {x_train_raw.min()} – {x_train_raw.max()}")
print(f"  Number of classes     : {len(np.unique(y_train_raw))}  ({np.unique(y_train_raw).tolist()})")
print("═" * 50)
print()
print("Answer: The MNIST training set contains 60,000 images, each 28×28 pixels (greyscale).")
print("        The test set contains 10,000 images of the same dimensions.")
print("        Each pixel value is an integer in the range [0, 255].")

---
## 1b. First 20 Values of Training Data
> **Question b:** What are the first 20 values of the training data?

In [ ]:
print("First 20 training labels (digit classes):")
print(y_train_raw[:20].tolist())
print()
print("First 20 training images flattened (first 20 pixel values of image 0):")
print(x_train_raw[0].flatten()[:20].tolist())

# Visualise the first 20 training images
fig, axes = plt.subplots(2, 10, figsize=(14, 3.2))
fig.patch.set_facecolor('#0D0F18')
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train_raw[i], cmap='gray')
    ax.set_title(str(y_train_raw[i]), color='#4F8EF7', fontsize=10, pad=2)
    ax.axis('off')
plt.suptitle('First 20 MNIST Training Images with Labels', color='white', fontsize=12, y=1.02)
plt.tight_layout(pad=0.3)
plt.show()

---
## 1c. Effect of Normalisation
> **Question c:** Does normalising affect the accuracy of the ANN? Explain your answer.

In [ ]:
def build_model(h1=256, h2=128, h3=64, act='relu', dropout=0.3):
    """Build an ANN with BatchNormalization for better generalisation."""
    model = Sequential([
        Dense(h1, activation=act, input_shape=(784,)),
        BatchNormalization(),
        Dropout(dropout),
        Dense(h2, activation=act),
        BatchNormalization(),
        Dropout(dropout),
        Dense(h3, activation=act),
        Dropout(dropout / 2),
        Dense(10, activation='softmax')
    ])
    model.compile(optimizer=Adam(0.001),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# ── Prepare labels (one-hot encoded) ─────────────────────────────────────────
y_train_cat = to_categorical(y_train_raw, 10)
y_test_cat  = to_categorical(y_test_raw,  10)

# ── Flatten & normalise ───────────────────────────────────────────────────────
x_train_raw_flat = x_train_raw.reshape(-1, 784).astype('float32')  # FIX: removed hardcoded 60000/10000
x_test_raw_flat  = x_test_raw.reshape(-1, 784).astype('float32')

print("Training WITHOUT normalisation (pixels 0–255)...")
m_no_norm = build_model()
m_no_norm.fit(x_train_raw_flat, y_train_cat, epochs=5, batch_size=64,
              validation_split=0.1, verbose=0)
_, acc_no_norm = m_no_norm.evaluate(x_test_raw_flat, y_test_cat, verbose=0)
print(f"  Test accuracy WITHOUT normalisation: {acc_no_norm*100:.2f}%")

x_train = x_train_raw_flat / 255.0
x_test  = x_test_raw_flat  / 255.0

# ── Augment training data with noise + small shifts (helps with bad writing) ──
def augment(x, n_extra=30000):
    """Add noise-corrupted and slightly-shifted copies to training data."""
    idx  = np.random.choice(len(x), n_extra, replace=True)
    imgs = x[idx].reshape(-1, 28, 28)

    # Gaussian noise
    noisy = np.clip(imgs + np.random.normal(0, 0.08, imgs.shape), 0, 1)

    # Random shift ±2px
    shifted = np.array([
        ndimage.shift(im, shift=(np.random.randint(-2, 3), np.random.randint(-2, 3)),
                      mode='nearest')
        for im in imgs
    ])

    augmented = np.vstack([noisy, shifted]).reshape(-1, 784)
    labels    = np.tile(y_train_cat[idx], (2, 1))
    return augmented, labels

print("\nAugmenting training data with noise + shifts...")
x_aug, y_aug = augment(x_train)
x_train_aug  = np.vstack([x_train, x_aug])
y_train_aug  = np.vstack([y_train_cat, y_aug])
print(f"  Extended dataset: {x_train_aug.shape[0]:,} samples")

print("\nTraining WITH normalisation (pixels 0.0–1.0)...")
m_norm = build_model()
m_norm.fit(x_train, y_train_cat, epochs=5, batch_size=64,
           validation_split=0.1, verbose=0)
_, acc_norm = m_norm.evaluate(x_test, y_test_cat, verbose=0)
print(f"  Test accuracy WITH normalisation   : {acc_norm*100:.2f}%")

print(f"\nImprovement from normalisation: +{(acc_norm - acc_no_norm)*100:.2f}%")
print()
print("Answer: Yes, normalisation significantly improves accuracy. Dividing by 255")
print("scales all pixel values to [0.0, 1.0], which prevents large gradient values")
print("from causing unstable weight updates during backpropagation.")


---
## 1d. What is a Densely / Fully Connected Layer?
> **Question d:** Explain what a densely or fully connected layer is.

In [ ]:
print("Answer (Question d):")
print("═" * 60)
print("A densely connected (or fully connected) layer is one where")
print("every neuron in that layer is connected to every neuron in")
print("the previous layer via a weighted edge.")
print()
print("Mathematically: output = activation(W · input + b)")
print("  W = weight matrix  (shape: [n_out, n_in])")
print("  b = bias vector    (shape: [n_out])")
print()
print("In this network:")
print("  Dense(128) — each of 128 neurons connects to all 784 inputs")
print("               → 784×128 + 128 = 100,480 parameters")
print("  Dense(64)  — each of 64 neurons connects to all 128 neurons")
print("               → 128×64 + 64 = 8,256 parameters")
print("  Dense(10)  — output layer, 10 neurons (one per digit class)")
print("               → 64×10 + 10 = 650 parameters")

# Show parameter counts
demo_model = build_model()
demo_model.summary()

---
## 1e. Why 784 Input Neurons but only 10 Output Neurons?
> **Question e:** Why does the ANN have 784 input neurons but only 10 output neurons?

In [ ]:
print("Answer (Question e):")
print("═" * 60)
print()
print("INPUT — 784 neurons:")
print("  Each MNIST image is 28×28 pixels = 784 pixels total.")
print("  The image is flattened into a 1D vector of 784 values.")
print("  Each input neuron receives exactly one pixel intensity value.")
print(f"  28 × 28 = {28*28} ✓")
print()
print("OUTPUT — 10 neurons:")
print("  There are exactly 10 possible digit classes: 0, 1, 2, ..., 9.")
print("  Each output neuron produces the probability that the input")
print("  image belongs to that class (using softmax activation).")
print("  The predicted digit = the output neuron with the highest probability.")
print()
print("  Example output: [0.01, 0.02, 0.89, 0.01, ...]")
print("                    0     1    ↑2     3  ...  → predicted digit = 2")

# Show example
sample = x_train[0:1]
probs  = m_norm.predict(sample, verbose=0)[0]
print(f"\nExample prediction probabilities for training image 0 (label={y_train_raw[0]}):")
for i, p in enumerate(probs):
    bar = '█' * int(p * 30)
    print(f"  Digit {i}: {bar:<30} {p*100:.2f}%")
print(f"  → Predicted: {np.argmax(probs)}")

---
## 1f. Parameter Experiments
> **Question f:** How do activation/loss functions, optimiser, layers, learning rate, and epochs affect accuracy?

### 1f-i. Effect of Epochs

In [ ]:
print("Experiment 1: Varying Number of Epochs (all else constant)")
print("  Base config: Adam lr=0.001, relu, 128→64→10, batch=32")
print("═" * 55)

epoch_values = [3, 5, 10, 15, 20]
epoch_results = []

for ep in epoch_values:
    m = build_model()
    m.fit(x_train, y_train_cat, epochs=ep, batch_size=32,
          validation_split=0.2, verbose=0)
    _, acc = m.evaluate(x_test, y_test_cat, verbose=0)
    epoch_results.append(acc)
    print(f"  Epochs = {ep:2d}  →  Test accuracy: {acc*100:.2f}%")

print(f"\n  Best: {epoch_values[np.argmax(epoch_results)]} epochs → {max(epoch_results)*100:.2f}%")

### 1f-ii. Effect of Learning Rate

In [ ]:
print("Experiment 2: Varying Learning Rate (all else constant)")
print("  Base config: Adam, relu, 128→64→10, batch=32, epochs=10")
print("═" * 55)

lr_values  = [0.1, 0.01, 0.001, 0.0001, 0.00001]
lr_results = []

for lr in lr_values:
    m = Sequential([
        Dense(128, activation='relu', input_shape=(784,)),
        Dropout(0.2),
        Dense(64,  activation='relu'),
        Dropout(0.2),
        Dense(10,  activation='softmax')
    ])
    m.compile(optimizer=Adam(lr), loss='categorical_crossentropy', metrics=['accuracy'])
    m.fit(x_train, y_train_cat, epochs=10, batch_size=32,
          validation_split=0.2, verbose=0)
    _, acc = m.evaluate(x_test, y_test_cat, verbose=0)
    lr_results.append(acc)
    print(f"  LR = {lr:<10}  →  Test accuracy: {acc*100:.2f}%")

print(f"\n  Best: lr={lr_values[np.argmax(lr_results)]} → {max(lr_results)*100:.2f}%")

### 1f-iii. Effect of Activation Function

In [ ]:
print("Experiment 3: Varying Activation Function (all else constant)")
print("  Base config: Adam lr=0.001, 128→64→10, batch=32, epochs=10")
print("═" * 55)

act_values  = ['relu', 'sigmoid', 'tanh', 'elu', 'selu']
act_results = []

for act in act_values:
    m = build_model(act=act)
    m.fit(x_train, y_train_cat, epochs=10, batch_size=32,
          validation_split=0.2, verbose=0)
    _, acc = m.evaluate(x_test, y_test_cat, verbose=0)
    act_results.append(acc)
    print(f"  Activation = {act:<8}  →  Test accuracy: {acc*100:.2f}%")

print(f"\n  Best: {act_values[np.argmax(act_results)]} → {max(act_results)*100:.2f}%")

### 1f-iv. Effect of Optimiser

In [ ]:
print("Experiment 4: Varying Optimiser (all else constant)")
print("  Base config: relu, 128→64→10, batch=32, epochs=10, lr=0.001")
print("═" * 55)

optimisers     = [Adam(0.001), SGD(0.001, momentum=0.9), RMSprop(0.001)]
opt_names      = ['Adam', 'SGD (momentum=0.9)', 'RMSprop']
opt_results    = []

for opt, name in zip(optimisers, opt_names):
    m = Sequential([
        Dense(128, activation='relu', input_shape=(784,)),
        Dropout(0.2),
        Dense(64,  activation='relu'),
        Dropout(0.2),
        Dense(10,  activation='softmax')
    ])
    m.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])
    m.fit(x_train, y_train_cat, epochs=10, batch_size=32,
          validation_split=0.2, verbose=0)
    _, acc = m.evaluate(x_test, y_test_cat, verbose=0)
    opt_results.append(acc)
    print(f"  Optimiser = {name:<22}  →  {acc*100:.2f}%")

print(f"\n  Best: {opt_names[np.argmax(opt_results)]} → {max(opt_results)*100:.2f}%")

### 1f-v. Effect of Loss Function

In [ ]:
print("Experiment 5: Varying Loss Function (all else constant)")
print("  Base config: Adam lr=0.001, relu, 128→64→10, batch=32, epochs=10")
print("═" * 55)

loss_values  = ['categorical_crossentropy', 'mean_squared_error', 'kullback_leibler_divergence']
loss_labels  = ['crossentropy', 'MSE', 'KLD']
loss_results = []

for loss, label in zip(loss_values, loss_labels):
    m = Sequential([
        Dense(128, activation='relu', input_shape=(784,)),
        Dropout(0.2),
        Dense(64,  activation='relu'),
        Dropout(0.2),
        Dense(10,  activation='softmax')
    ])
    m.compile(optimizer=Adam(0.001), loss=loss, metrics=['accuracy'])
    m.fit(x_train, y_train_cat, epochs=10, batch_size=32,
          validation_split=0.2, verbose=0)
    _, acc = m.evaluate(x_test, y_test_cat, verbose=0)
    loss_results.append(acc)
    print(f"  Loss = {label:<14}  →  {acc*100:.2f}%")

print(f"\n  Best: {loss_labels[np.argmax(loss_results)]} → {max(loss_results)*100:.2f}%")

### 1f-vi. Effect of Number of Layers

In [ ]:
print("Experiment 6: Varying Number of Hidden Layers (all else constant)")
print("  Base config: Adam lr=0.001, relu, batch=64, epochs=10")
print("═" * 55)

# FIX: define configs as callables so layers are freshly instantiated each run
layer_configs = [
    (lambda: [Dense(128, activation='relu', input_shape=(784,)), Dropout(0.2),
              BatchNormalization(),
              Dense(10, activation='softmax')],
     '1 hidden (128)'),
    (lambda: [Dense(128, activation='relu', input_shape=(784,)), Dropout(0.2),
              BatchNormalization(),
              Dense(64, activation='relu'),   Dropout(0.2),
              Dense(10, activation='softmax')],
     '2 hidden (128→64)'),
    (lambda: [Dense(128, activation='relu', input_shape=(784,)), Dropout(0.2),
              BatchNormalization(),
              Dense(64, activation='relu'),   Dropout(0.2),
              Dense(32, activation='relu'),
              Dense(10, activation='softmax')],
     '3 hidden (128→64→32)'),
]
layer_results = []

for build_layers, name in layer_configs:
    m = Sequential(build_layers())      # FIX: call lambda to get fresh layer instances
    m.compile(optimizer=Adam(0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    m.fit(x_train, y_train_cat, epochs=10, batch_size=64,
          validation_split=0.1, verbose=0)
    _, acc = m.evaluate(x_test, y_test_cat, verbose=0)
    layer_results.append(acc)
    print(f"  {name:<28}  →  {acc*100:.2f}%")

print(f"\n  Best: {[n for _,n in layer_configs][np.argmax(layer_results)]} → {max(layer_results)*100:.2f}%")


### Experiment Summary Plot

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.patch.set_facecolor('#0D0F18')
fig.suptitle('Effect of Each Parameter on Test Accuracy (Question f)',
             color='white', fontsize=14, fontweight='bold', y=1.01)

experiments = [
    ([str(v) for v in epoch_values], epoch_results,  'Epochs',             'Number of Epochs'),
    ([str(v) for v in lr_values],    lr_results,     'Learning Rate',      'Learning Rate'),
    (act_values,                     act_results,    'Activation Function', 'Activation'),
    (opt_names,                      opt_results,    'Optimiser',           'Optimiser'),
    (loss_labels,                    loss_results,   'Loss Function',       'Loss Function'),
    ([n for _,n in layer_configs],   layer_results,  'Number of Layers',    'Layer Config'),  # FIX: unpack (fn, name) tuples
]

for ax, (xlabels, results, title, xlabel) in zip(axes.flat, experiments):
    ax.set_facecolor('#161926')
    colors = ['#34D399' if r == max(results) else '#4F8EF7' for r in results]
    bars = ax.bar(range(len(results)), [r*100 for r in results],
                  color=colors, edgecolor='#2A2E45', linewidth=0.8)
    ax.axhline(97, color='#34D399', lw=1.2, ls='--', alpha=0.8, label='97%')
    ax.axhline(90, color='#FBBF24', lw=1.0, ls=':',  alpha=0.7, label='90%')
    for bar, r in zip(bars, results):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                f'{r*100:.1f}%', ha='center', va='bottom', fontsize=7, color='white')
    ax.set_xticks(range(len(xlabels)))
    ax.set_xticklabels(xlabels, rotation=20, ha='right', fontsize=8, color='#8B91A8')
    ax.set_title(title, color='white', fontsize=10, fontweight='bold')
    ax.set_xlabel(xlabel, color='#8B91A8', fontsize=8)
    ax.set_ylabel('Accuracy (%)', color='#8B91A8', fontsize=8)
    ax.tick_params(colors='#8B91A8', labelsize=8)
    ax.set_ylim(max(0, min(results)*100 - 15), 100)
    ax.legend(fontsize=7, labelcolor='white', facecolor='#1F2235', edgecolor='#2A2E45')
    for sp in ax.spines.values(): sp.set_color('#2A2E45')
    ax.grid(axis='y', color='#2A2E45', ls='--', lw=0.5)

plt.tight_layout()
plt.savefig('experiment_results.png', dpi=150, bbox_inches='tight', facecolor='#0D0F18')
plt.show()
print("Plot saved as experiment_results.png")


---
## 1g. Confusion Matrix
> **Question g:** What is the purpose of the confusion matrix and what insight can it provide?

In [ ]:
# Train the final best model on augmented data for robustness to bad handwriting
print("Training final model on augmented dataset...")
final_model = Sequential([
    Dense(512, activation='relu', input_shape=(784,)),
    BatchNormalization(),
    Dropout(0.4),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(64,  activation='relu'),
    Dense(10,  activation='softmax')
])
final_model.compile(optimizer=Adam(0.001),
                    loss='categorical_crossentropy',
                    metrics=['accuracy'])

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
]

history = final_model.fit(
    x_train_aug, y_train_aug,           # FIX: use augmented data
    epochs=30, batch_size=64,
    validation_data=(x_test, y_test_cat),
    callbacks=callbacks,
    verbose=1
)

_, final_acc = final_model.evaluate(x_test, y_test_cat, verbose=0)
print(f"\nFinal model test accuracy: {final_acc*100:.2f}%")


In [ ]:
# Generate predictions
y_pred_probs = final_model.predict(x_test, verbose=0)
y_pred       = np.argmax(y_pred_probs, axis=1)
y_true       = np.argmax(y_test_cat,  axis=1)

cm = confusion_matrix(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0D0F18')

# Raw counts
ax1 = axes[0]
ax1.set_facecolor('#161926')
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10), ax=ax1)
ax1.set_title('Confusion Matrix (counts)', color='white', fontsize=12, pad=10)
ax1.set_xlabel('Predicted digit', color='#8B91A8')
ax1.set_ylabel('Actual digit',    color='#8B91A8')
ax1.tick_params(colors='#8B91A8')

# Normalised
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
ax2 = axes[1]
ax2.set_facecolor('#161926')
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10), ax=ax2)
ax2.set_title('Confusion Matrix (normalised)', color='white', fontsize=12, pad=10)
ax2.set_xlabel('Predicted digit', color='#8B91A8')
ax2.set_ylabel('Actual digit',    color='#8B91A8')
ax2.tick_params(colors='#8B91A8')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight', facecolor='#0D0F18')
plt.show()

print("Answer (Question g):")
print("═" * 60)
print("A confusion matrix shows — for each actual class — how many")
print("predictions were made for each possible class.")
print("  • Diagonal = correct predictions")
print("  • Off-diagonal = misclassifications")
print()
print("Insights it provides:")
print("  1. Which digits are most often confused with each other")
print("     (e.g., 4 vs 9, or 3 vs 8 — visually similar)")
print("  2. Whether the model is biased toward predicting certain classes")
print("  3. Per-class accuracy, not just overall accuracy")
print("  4. Helps guide improvements (e.g., collect more data for weak classes)")

# Show classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=[str(i) for i in range(10)]))

---
## 2. Training History Plot

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0D0F18')

epochs_range = range(1, len(history.history['accuracy']) + 1)

for ax, train_key, val_key, title, ylabel, c1, c2 in [
    (ax1, 'accuracy', 'val_accuracy', 'Accuracy', 'Accuracy',  '#4F8EF7', '#A78BFA'),
    (ax2, 'loss',     'val_loss',     'Loss',      'Loss',      '#F87171', '#FBBF24'),
]:
    ax.set_facecolor('#161926')
    ax.plot(epochs_range, history.history[train_key], color=c1, lw=2, label='Train')
    ax.plot(epochs_range, history.history[val_key],   color=c2, lw=2, ls='--', label='Validation')
    if train_key == 'accuracy':
        ax.axhline(0.97, color='#34D399', lw=1, ls=':', alpha=0.8, label='97% target')
    ax.set_title(title, color='white', fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch', color='#8B91A8')
    ax.set_ylabel(ylabel, color='#8B91A8')           # FIX: was missing y-axis label
    ax.tick_params(colors='#8B91A8')
    ax.legend(fontsize=9, labelcolor='white', facecolor='#1F2235', edgecolor='#2A2E45')
    for sp in ax.spines.values(): sp.set_color('#2A2E45')
    ax.grid(color='#2A2E45', ls='--', lw=0.5)

plt.suptitle('Final Model Training History', color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight', facecolor='#0D0F18')
plt.show()


---
## 3. Part 2 — Predict Own Hand-Drawn Images
Draw digits in MS Paint (28×28 or larger), save as JPEG. Place them in a folder and update the path below.

In [ ]:
import os
from PIL import Image, ImageOps, ImageFilter

def centre_digit(arr_28x28):
    """
    Crop tight around the digit and re-centre it, matching how MNIST was prepared.
    Uses centre-of-mass (scipy) so even off-centre or messy digits are aligned.
    """
    cy, cx = ndimage.center_of_mass(arr_28x28)
    shift_y = 14 - int(round(cy))
    shift_x = 14 - int(round(cx))
    return ndimage.shift(arr_28x28, shift=(shift_y, shift_x), mode='constant', cval=0)

def preprocess_own_image(path):
    """
    Load a hand-drawn digit image and preprocess it to match MNIST format.

    Fixes vs original:
      1. AUTO-INVERT  — MNIST is white-on-black; Paint/photos are black-on-white.
                        We detect background colour and invert if needed.
      2. CENTRE       — shift digit to centre-of-mass position like MNIST digits.
      3. DENOISE      — apply a median filter to suppress scan/photo noise.
      4. CONTRAST     — stretch contrast so faint lines become solid strokes.
      5. ROBUST AXES  — handles any input dimensions without crashing.
    """
    img = Image.open(path).convert('L')               # greyscale

    # ── 3. Denoise (median filter removes salt-and-pepper noise) ───────────────
    img = img.filter(ImageFilter.MedianFilter(size=3))

    # ── 4. Contrast stretch ─────────────────────────────────────────────────────
    img = ImageOps.autocontrast(img, cutoff=2)         # clip 2% outliers each end

    img = img.resize((28, 28), Image.LANCZOS)          # resize AFTER denoising
    arr = np.array(img, dtype='float32') / 255.0       # normalise to [0, 1]

    # ── 1. Auto-invert: background should be 0 (black), digit should be 1 (white)
    # If the image mean > 0.5 it is mostly white (black-on-white) → invert it.
    if arr.mean() > 0.5:
        arr = 1.0 - arr

    # ── 2. Centre digit using centre-of-mass ────────────────────────────────────
    arr = centre_digit(arr)

    return arr.reshape(1, 784)

# ── Set your own images folder path here ──────────────────────────────────────
OWN_IMAGES_FOLDER = './own_digits'  # <-- Change this to your folder

if os.path.exists(OWN_IMAGES_FOLDER):
    image_files = [f for f in sorted(os.listdir(OWN_IMAGES_FOLDER))
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    print(f"Found {len(image_files)} image(s) in '{OWN_IMAGES_FOLDER}'")
    print("═" * 55)

    results = []
    images  = []

    for fname in image_files:
        path  = os.path.join(OWN_IMAGES_FOLDER, fname)
        arr   = preprocess_own_image(path)
        probs = final_model.predict(arr, verbose=0)[0]
        pred  = np.argmax(probs)
        conf  = probs[pred] * 100
        results.append((fname, pred, conf))
        images.append(arr.reshape(28, 28))
        print(f"  {fname:<30}  →  Digit: {pred}  (confidence: {conf:.1f}%)")

    if results:
        n    = len(images)
        cols = min(n, 10)
        rows = (n + cols - 1) // cols

        # FIX: flatten axes safely regardless of rows/cols dimensions
        fig, axes_raw = plt.subplots(rows, cols, figsize=(cols * 1.6, rows * 2))
        fig.patch.set_facecolor('#0D0F18')
        axes_flat = np.array(axes_raw).flatten()      # works for any shape

        for ax, img, (fname, pred, conf) in zip(axes_flat, images, results):
            ax.imshow(img, cmap='gray')
            ax.set_title(f'Pred: {pred}\n{conf:.0f}%',
                         color='#34D399' if conf > 70 else '#FBBF24', fontsize=8)
            ax.axis('off')

        # Hide any leftover empty axes
        for ax in axes_flat[len(images):]:
            ax.axis('off')

        plt.suptitle('Own Hand-Drawn Digit Predictions', color='white',
                     fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.savefig('own_digit_predictions.png', dpi=150,
                    bbox_inches='tight', facecolor='#0D0F18')
        plt.show()
else:
    print(f"Folder '{OWN_IMAGES_FOLDER}' not found.")
    print("Create the folder, add your hand-drawn JPEG digit images, and re-run this cell.")
    print("Tip: Draw each digit (0–9) twice in MS Paint, save as digit_0a.jpg, digit_0b.jpg, etc.")
    print()
    print("Preprocessing pipeline applied to your images:")
    print("  1. Convert to greyscale")
    print("  2. Median filter  — removes scan/camera noise")
    print("  3. Auto-contrast  — makes faint strokes solid")
    print("  4. Resize to 28×28")
    print("  5. Auto-invert    — black-on-white → white-on-black (MNIST style)")
    print("  6. Centre-of-mass — aligns digit to centre like MNIST training data")


---
## Summary

| Question | Topic | Answer |
|----------|-------|--------|
| a | MNIST dimensions | 60,000 train (28×28×1), 10,000 test |
| b | First 20 values | See cell above |
| c | Normalisation effect | Yes — scales gradients, improves convergence |
| d | Dense layer | Every neuron connected to every neuron in previous layer |
| e | 784 in / 10 out | 28×28=784 pixels input; 10 digit classes output |
| f | Parameter effects | See 6 experiments above |
| g | Confusion matrix | Shows per-class predictions; reveals misclassification patterns |